# Paper 2 · Langkah 10: audit kompresi seluruh AI-ArtBench (CPU)

Sejauh ini selisih kualitas JPEG (karya manusia 95, gambar AI 75) baru dibuktikan pada **sampel 3.000 gambar**. Notebook ini memeriksa **seluruh 185.015 file** di dataset, supaya klaimnya berlaku untuk dataset, bukan untuk sampel.

Yang dikerjakan: mengunduh dataset dari Kaggle (sama seperti langkah 1), lalu membaca metadata tiap file: format, ukuran, dimensi, kualitas JPEG dari tabel kuantisasi, subsampling warna, progresif, dan EXIF. **Tidak ada model yang dilatih dan tidak ada gambar yang disimpan.**

**Aturan baca yang dikunci sebelum hasil:** klaim "berlaku untuk seluruh dataset" dipakai kalau, di split train **dan** test, minimal 99% file karya manusia punya kualitas 90 atau lebih dan minimal 99% file gambar AI punya kualitas 80 atau kurang. Kalau tidak, naskah melaporkan persentase sebenarnya beserta pengecualiannya.

**Sebelum Run all:** runtime **CPU** sudah cukup. Siapkan file `kaggle.json` yang dulu dipakai di langkah 1; sel 2 akan memintanya. Perkiraan 15 sampai 25 menit, sebagian besar untuk mengunduh 11 GB.

## 1. Persiapan

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, time, hashlib, platform
import numpy as np, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import PIL
from PIL import Image, JpegImagePlugin

PROJECT = Path('/content/drive/MyDrive/Paper2')
NOTES, RESULTS = PROJECT / 'catatan', PROJECT / 'hasil'
ROOT = Path('/content/aiartbench')
env = (f'tanggal: {time.strftime("%Y-%m-%d %H:%M")}\npython: {platform.python_version()}\nnumpy: {np.__version__}\n'
       f'pandas: {pd.__version__}\npillow: {PIL.__version__}\ndataset: kaggle ravidussilva/real-ai-art\n')
(NOTES / 'lingkungan_langkah10.txt').write_text(env)
print(env)

## 2. Unduh dataset dari Kaggle

Saat diminta, pilih file `kaggle.json`. Kalau dataset sudah ada di sesi ini, pengunduhan dilewati.

In [ ]:
if not ROOT.exists() or not any(ROOT.iterdir()):
    !pip -q install kaggle
    from google.colab import files
    uploaded = files.upload()   # pilih kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
    !kaggle datasets download -d ravidussilva/real-ai-art -p /content/aiartbench --unzip -q
else:
    print('Dataset sudah ada di sesi ini, pengunduhan dilewati.')
!du -sh /content/aiartbench

## 3. Daftar seluruh file

In [ ]:
EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif', '.tif', '.tiff'}
rows = []
for p in ROOT.rglob('*'):
    if p.is_file() and p.suffix.lower() in EXT:
        rel = p.relative_to(ROOT).parts
        rows.append({'path': str(p), 'folder': p.parent.name, 'parts': '/'.join(rel[:-1]), 'ext': p.suffix.lower()})
inv = pd.DataFrame(rows)

def parse(row):
    f = row['folder']; fl = f.lower()
    source = 'latent_diffusion' if 'ai_ld' in fl else ('stable_diffusion' if 'ai_sd' in fl else 'human')
    style = re.sub(r'(?i)^ai_(ld|sd)_', '', f).strip().lower().replace('-', '_')
    parts = row['parts'].lower().split('/')
    split = 'train' if 'train' in parts else ('test' if 'test' in parts else 'unknown')
    return pd.Series({'source': source, 'style': style, 'split': split})
inv = pd.concat([inv, inv.apply(parse, axis=1)], axis=1)
print('Total file gambar:', len(inv))
print(inv.pivot_table(index='source', columns='split', values='path', aggfunc='count', fill_value=0, margins=True).to_string())
print('\nEkstensi file:', inv.ext.value_counts().to_dict())
assert inv.source.nunique() == 3 and inv['style'].nunique() == 10, 'Struktur folder tidak seperti yang diharapkan. Kirim hasil sel ini ke Claude.'

## 4. Baca metadata tiap file

Hanya kepala file yang dibaca, jadi cepat. Hasil per file disimpan ke Drive dalam bentuk terkompresi.

In [ ]:
STD_LUMA = np.array([
    16,11,10,16,24,40,51,61, 12,12,14,19,26,58,60,55, 14,13,16,24,40,57,69,56, 14,17,22,29,51,87,80,62,
    18,22,37,56,68,109,103,77, 24,35,55,64,81,104,113,92, 49,64,78,87,103,121,120,101, 72,92,95,98,112,100,103,99])

def file_info(path):
    out = {'file_size_bytes': os.path.getsize(path), 'format': None, 'width': None, 'height': None, 'mode': None,
           'jpeg_quality_est': np.nan, 'qtable_id': None, 'subsampling': None, 'progressive': None, 'has_exif': None, 'error': None}
    try:
        with Image.open(path) as im:
            out.update(format=im.format, width=im.size[0], height=im.size[1], mode=im.mode)
            if im.format == 'JPEG':
                q = getattr(im, 'quantization', None) or {}
                luma = np.array(q.get(0, []), dtype=float)
                if luma.size == 64:
                    scale = 100.0 * luma.sum() / STD_LUMA.sum()
                    out['jpeg_quality_est'] = round(float((200 - scale) / 2 if scale <= 100 else 5000 / scale), 1)
                out['qtable_id'] = hashlib.md5(repr(sorted((k, tuple(v)) for k, v in q.items())).encode()).hexdigest()[:8]
                try: sub = JpegImagePlugin.get_sampling(im)
                except Exception: sub = -1
                out['subsampling'] = {0: '4:4:4', 1: '4:2:2', 2: '4:2:0'}.get(sub, 'lain')
                out['progressive'] = int(bool(im.info.get('progressive', 0) or im.info.get('progression', 0)))
                out['has_exif'] = int('exif' in im.info)
    except Exception as e:
        out['error'] = type(e).__name__
    return out

per_file = RESULTS / 'audit_ai_artbench_seluruh_file.csv.gz'
if per_file.exists():
    A = pd.read_csv(per_file); print('Metadata sudah pernah dibaca, memakai file tersimpan:', len(A), 'baris')
else:
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex:
        info = list(ex.map(file_info, inv['path']))
    A = pd.concat([inv[['source', 'style', 'split', 'folder']].reset_index(drop=True), pd.DataFrame(info)], axis=1)
    A.insert(0, 'file', [Path(p).name for p in inv['path']])
    A['dimensi'] = A.width.astype('Int64').astype(str) + ' x ' + A.height.astype('Int64').astype(str)
    A.to_csv(per_file, index=False, compression='gzip')
    print(f'Metadata {len(A)} file dibaca dalam {time.time() - t0:.0f} detik. Tersimpan:', per_file.name)
print('File yang tidak bisa dibaca:', int(A.error.notna().sum()))

## 5. Ringkasan

- **Q1**: kualitas JPEG per sumber dan split.
- **Q2**: sebaran nilai kualitas (semua nilai yang muncul).
- **Q3**: kualitas per gaya.
- **Q4**: tabel kuantisasi, subsampling, progresif, EXIF, format.
- **Q5**: dimensi dan ukuran file.
- **Q6**: seberapa baik kualitas JPEG saja menebak kelas.
- **Q7**: pemeriksaan aturan baca dan perbandingan dengan sampel 3.000.

In [ ]:
SRC = {'human': 'manusia', 'latent_diffusion': 'LD', 'stable_diffusion': 'SD'}
A['sumber'] = A.source.map(SRC); A['is_ai'] = (A.source != 'human').astype(int)
J = A[A.format == 'JPEG']

print('Q1. KUALITAS JPEG (perkiraan dari tabel kuantisasi) per sumber dan split')
q1 = J.groupby(['sumber', 'split']).jpeg_quality_est.agg(n='size', minimum='min', median='median', maksimum='max', nilai_unik='nunique')
print(q1.to_string()); q1.to_csv(RESULTS / 'audit_kualitas_per_sumber.csv')

print('\nQ2. SEBARAN NILAI KUALITAS per sumber (jumlah file)')
q2 = J.groupby(['sumber', 'jpeg_quality_est']).size().rename('jumlah').reset_index()
print(q2.to_string(index=False)); q2.to_csv(RESULTS / 'audit_sebaran_kualitas.csv', index=False)

print('\nQ3. KUALITAS per gaya (minimum sampai maksimum)')
q3 = J.groupby(['style', 'sumber']).jpeg_quality_est.agg(['min', 'max']).apply(lambda r: f"{r['min']:g}" if r['min'] == r['max'] else f"{r['min']:g} sampai {r['max']:g}", axis=1).unstack()
print(q3.to_string())

print('\nQ4. TABEL KUANTISASI, SUBSAMPLING, PROGRESIF, EXIF, FORMAT')
print('   Format file per sumber:'); print(A.groupby(['sumber', 'format'], dropna=False).size().unstack(fill_value=0).to_string())
print('   Jumlah tabel kuantisasi berbeda per sumber:', J.groupby('sumber').qtable_id.nunique().to_dict())
print('   Tabel kuantisasi yang dipakai lebih dari satu sumber:', int((J.groupby('qtable_id').sumber.nunique() > 1).sum()))
print('   Subsampling:'); print(J.groupby(['sumber', 'subsampling']).size().unstack(fill_value=0).to_string())
print('   Progresif (jumlah file):', J.groupby('sumber').progressive.sum().astype(int).to_dict(), '| EXIF (jumlah file):', J.groupby('sumber').has_exif.sum().astype(int).to_dict())

print('\nQ5. DIMENSI DAN UKURAN FILE')
print(A.groupby(['sumber', 'dimensi']).size().rename('jumlah').reset_index().sort_values(['sumber', 'jumlah'], ascending=[True, False]).groupby('sumber').head(4).to_string(index=False))
print('   Median ukuran file (KB):', (A.groupby('sumber').file_size_bytes.median() / 1024).round(1).to_dict())

print('\nQ6. KUALITAS JPEG SAJA SEBAGAI PENEBAK KELAS (aturan: kualitas 85 atau lebih = karya manusia)')
pred_human = J.jpeg_quality_est >= 85
acc = ((pred_human & (J.is_ai == 0)) | (~pred_human & (J.is_ai == 1))).mean()
print(f'   Akurasi pada {len(J)} file JPEG: {100 * acc:.3f}%')
for sp, g in J.groupby('split'):
    ph = g.jpeg_quality_est >= 85
    print(f"   split {sp}: akurasi {100 * ((ph & (g.is_ai == 0)) | (~ph & (g.is_ai == 1))).mean():.3f}% ({len(g)} file)")
wrong = J[(pred_human & (J.is_ai == 1)) | (~pred_human & (J.is_ai == 0))]
print('   File yang salah ditebak oleh aturan ini:', len(wrong))
if len(wrong): print(wrong.groupby(['sumber', 'split', 'style', 'jpeg_quality_est']).size().rename('jumlah').reset_index().head(30).to_string(index=False))

print('\nQ7. ATURAN BACA')
rows = []
for sp, g in J.groupby('split'):
    h, a = g[g.is_ai == 0], g[g.is_ai == 1]
    rows.append({'split': sp, 'manusia: % kualitas >= 90': 100 * (h.jpeg_quality_est >= 90).mean(), 'AI: % kualitas <= 80': 100 * (a.jpeg_quality_est <= 80).mean(), 'n manusia': len(h), 'n AI': len(a)})
q7 = pd.DataFrame(rows).set_index('split'); print(q7.round(3).to_string()); q7.to_csv(RESULTS / 'audit_aturan_baca.csv')
ok = bool((q7['manusia: % kualitas >= 90'] >= 99).all() and (q7['AI: % kualitas <= 80'] >= 99).all() and len(J) == len(A))
print(f"   Kesimpulan: {'BERLAKU UNTUK SELURUH DATASET' if ok else 'ADA PENGECUALIAN, laporkan persentase sebenarnya'}")
if len(J) != len(A): print('   Catatan: tidak semua file berformat JPEG; lihat tabel format di Q4.')
print('\n   Pembanding dari sampel 3.000 (langkah 2): manusia 95; LD 74,8; SD 74,8; semua 4:2:0; tanpa progresif dan EXIF.')
print('\nTersimpan di', RESULTS)

## Selesai

Kalau tabel Q1 sampai Q7 muncul, semua beres. Tekan **Cmd+S**, lalu kirim link Colab-nya ke Claude.